# 02_預訓練yolo11n推論.py
**電腦視覺與深度學習 › Day20 YOLO11n 物件偵測(預訓練到自己訓練)**  ·  ⚠️ 錯誤示範(有 bug,對照修正版看)

這支程式用 Ultralytics 的 YOLO11n 預訓練模型做即時物件偵測,五分鐘就能看到成果。和 Day18 的 MediaPipe 一樣認得 COCO 80 類,但之後可以自己訓練。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/0)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> ⚡ 這支會用到深度學習,建議先到「執行階段 › 變更執行階段類型」選 **T4 GPU**。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY10"
!pip -q install ultralytics
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '02_預訓練yolo11n推論.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 說明與匯入
yolo11n.pt 是在 COCO 80 類上訓練好的,第一次執行會自動下載大約 5.5 MB。

In [ ]:
# 02 — YOLO11n 預訓練模型即時 webcam 偵測
#
# Ultralytics 官方的 yolo11n.pt 是在 COCO 80 類上訓練的（同 DAY9 MediaPipe）。
# 我們直接拿來玩，第一次執行會自動下載 ~5.5MB。
#
# 需要：pip install ultralytics opencv-python pillow

import os
import time
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

### 載入 YOLO
一行 YOLO 加上檔名就載入模型,印出類別數,應該是 80。

In [ ]:
from PIL import Image, ImageDraw, ImageFont

# ====== 載入模型 ======
# 第一次執行會自動下載到當前目錄
model = YOLO("yolo11n.pt")

### 中文字型與翻譯表
和 Day18 一樣,找中文字型,並準備 COCO 80 類的中文翻譯。

In [ ]:
print(f"模型類別數：{len(model.names)}")

# ====== 中文字型（cv2.putText 不支援 CJK）======
FONT_CANDIDATES = [
    "C:/Windows/Fonts/msjh.ttc",
    "C:/Windows/Fonts/msyh.ttc",
    "C:/Windows/Fonts/simhei.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)

# COCO 80 類的中文翻譯（同 DAY9 08）
COCO_ZH = {
    "person":"人", "bicycle":"腳踏車", "car":"汽車", "motorcycle":"機車",
    "airplane":"飛機", "bus":"公車", "train":"火車", "truck":"卡車",
    "boat":"船", "traffic light":"紅綠燈", "fire hydrant":"消防栓",
    "stop sign":"停止標誌", "parking meter":"停車計時器", "bench":"長椅",
    "bird":"鳥", "cat":"貓", "dog":"狗", "horse":"馬", "sheep":"羊",
    "cow":"牛", "elephant":"大象", "bear":"熊", "zebra":"斑馬",
    "giraffe":"長頸鹿", "backpack":"背包", "umbrella":"傘",
    "handbag":"手提包", "tie":"領帶", "suitcase":"行李箱", "frisbee":"飛盤",
    "skis":"滑雪板", "snowboard":"雪橇", "sports ball":"球", "kite":"風箏",
    "baseball bat":"球棒", "baseball glove":"棒球手套", "skateboard":"滑板",
    "surfboard":"衝浪板", "tennis racket":"網球拍", "bottle":"瓶子",
    "wine glass":"酒杯", "cup":"杯子", "fork":"叉子", "knife":"刀",
    "spoon":"湯匙", "bowl":"碗", "banana":"香蕉", "apple":"蘋果",
    "sandwich":"三明治", "orange":"橘子", "broccoli":"青花菜",
    "carrot":"紅蘿蔔", "hot dog":"熱狗", "pizza":"披薩", "donut":"甜甜圈",
    "cake":"蛋糕", "chair":"椅子", "couch":"沙發", "potted plant":"盆栽",
    "bed":"床", "dining table":"餐桌", "toilet":"馬桶", "tv":"電視",
    "laptop":"筆電", "mouse":"滑鼠", "remote":"遙控器", "keyboard":"鍵盤",
    "cell phone":"手機", "microwave":"微波爐", "oven":"烤箱",
    "toaster":"烤麵包機", "sink":"水槽", "refrigerator":"冰箱", "book":"書",
    "clock":"時鐘", "vase":"花瓶", "scissors":"剪刀", "teddy bear":"泰迪熊",
    "hair drier":"吹風機", "toothbrush":"牙刷",
}

### 依類別配色
用名稱的雜湊值配顏色,同一次執行裡同一類同一色。

In [ ]:
def 依類別配色(name):
    h = hash(name) & 0xFFFFFF
    return (h & 0xFF, (h >> 8) & 0xFF, (h >> 16) & 0xFF)

### 畫標籤
彩色底白字的標籤,做法和 MediaPipe 那支相同。

### 開鏡頭
打開鏡頭,準備記錄 FPS。

In [ ]:
def 畫中文標籤(img_bgr, text, xy, size, bg_bgr):
    """在指定位置畫「彩色底 + 白字」"""
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(FONT_PATH, size)
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    tw, th = r - l, b - t
    x, y = xy
    bg_rgb = (bg_bgr[2], bg_bgr[1], bg_bgr[0])
    draw.rectangle((x, y, x + tw + 6, y + th + 6), fill=bg_rgb)
    draw.text((x + 3, y + 3), text, font=font, fill=(255, 255, 255))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


# ====== 開 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

print("按 q 離開")

fps_hist = []

### ⚠️ 一行推論
model.predict 一行就完成偵測。imgsz 是送進模型的圖片大小,conf 是信心門檻。注意上面的註解說 320 就很夠,但程式其實寫 640,兩者對不起來;想要更快就改成 320 試試。

### 取出每個框
Ultralytics 已經幫我們做完去除重複框的 NMS。每個框有左上、右下座標,信心分數和類別編號;用編號查出英文名稱,再翻成中文,畫框和標籤。

### FPS 與收拾
計算並顯示平均 FPS,按 q 離開,最後收拾。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    t0 = time.time()

    # Ultralytics 一行推論；imgsz 越小越快，320 對即時應用很夠
    results = model.predict(frame, imgsz=640, conf=0.4, verbose=False)

    # 解析結果（Ultralytics 已幫我們做完 NMS）
    for r in results:
        boxes = r.boxes
        if boxes is None:
            continue
        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes.xyxy[i].tolist()
            conf = float(boxes.conf[i])
            cls_id = int(boxes.cls[i])
            name_en = model.names[cls_id]
            name_zh = COCO_ZH.get(name_en, name_en)
            color   = 依類別配色(name_en)

            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 3)
            frame = 畫中文標籤(frame, f"{name_zh} {conf*100:.0f}%",
                              (int(x1), max(0, int(y1) - 30)), 20, color)

    # FPS
    fps_hist.append(1.0 / max(time.time() - t0, 1e-3))
    if len(fps_hist) > 30:
        fps_hist.pop(0)
    fps = sum(fps_hist) / len(fps_hist)
    frame = 畫中文標籤(frame, f"FPS {fps:.1f}", (10, 10), 22, (30, 30, 30))

    cv2.imshow("YOLO11n (預訓練 COCO 80 類)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## 小結
預訓練模型五分鐘就能用。接下來我們要下載資料集、訓練一個只認你想要的東西的專屬模型。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/0) 看逐句拆解、做練習。